# 1 -  Análisis de sentimientos

**Objetivo**

Entrenar un modelo de clasificación que permita identificar si una reseña de película es positiva o negativa.

**Los datos**

Nuestro proyecto consistirá en entrenar un modelo de análisis de sentimiento para identificar reseñas positivas y negativas de películas. Utilizaremos una base de datos de 50,000 reseñas de películas en IMDB (25,000 positivas y 25,000 negativas).



## Importación de librerías y carga de datos

In [2]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

In [27]:
df = pd.read_csv("./data/IMDB Dataset.csv")

## Exploración de datos

Comenzamos inspeccionando los datos:

In [28]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 781.4 KB


Nuestro conjunto de datos tiene dos columnas:

Review:  Reseña que escribió alguna persona acerca de una película (no sabemos a qué película fue, pero no es relevante para este análisis).

Sentiment: Si la reseña fue positiva o negativa (negative o positive)

Finalmente, veamos cuántas reseñas positivas y cuántas negativas tenemos:

In [31]:
df.sentiment.value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Estamos por comenzar a hacer análisis de sentimiento usando Python. Nuestro modelo será capaz de determinar si una reseña es positiva o negativa. Por ejemplo, después de entrenar nuestro modelo, si le pedimos que analice el texto “I really loved this movie,” nuestro modelo debería indicarnos que se trata de una reseña positiva.

Esto podría dar la impresión de que la computadora está comprendiendo el texto o el lenguaje de la misma manera que lo haría un ser humano. Sin embargo, lo que haremos es algo que, aunque suene sorprendente, es fundamental en el procesamiento de lenguaje natural. Vamos a convertir los enunciados en una representación numérica. A esto se le conoce como vectorización del texto.

Mediante técnicas como TF-IDF, podemos transformar las palabras y frases en vectores, que son simplemente listas de números que representan la importancia de cada término en el contexto del texto. Estos vectores son los que el modelo utiliza para "comprender" el contenido y realizar predicciones sobre el sentimiento de la reseña.

Este proceso no implica comprensión del lenguaje en el sentido humano, sino una interpretación matemática que permite al modelo hacer inferencias basadas en patrones observados durante su entrenamiento.

## Preparación de datos

Este proceso de entrenamiento será mucho más pesado que otros que hemos visto anteriormente. Por lo tanto, para fines demostrativos, reduciremos el tamaño de nuestros datos. Tomaremos 5,000 reseñas positivas y 5,000 negativas

In [35]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

df_reviews.info()
df_reviews.sentiment.value_counts()

<class 'pandas.DataFrame'>
Index: 10000 entries, 0 to 10048
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     10000 non-null  str  
 1   sentiment  10000 non-null  str  
dtypes: str(2)
memory usage: 234.4 KB


sentiment
positive    5000
negative    5000
Name: count, dtype: int64

## División de datos

In [36]:
from sklearn.model_selection import train_test_split

train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)

train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']


# Imprimir shapes
print("train_x:", train_x.shape)
print("train_y:", train_y.shape)
print("test_x:", test_x.shape)
print("test_y:", test_y.shape)


train_x: (6700,)
train_y: (6700,)
test_x: (3300,)
test_y: (3300,)


Si inspeccionamos train_y, vemos:

In [37]:
train_y.value_counts()

sentiment
negative    3378
positive    3322
Name: count, dtype: int64

## De texto a números

Ahora procederemos a transformar nuestros valores de texto a valores numéricos. Para esto, usaremos Scikit Learn. Específicamente, utilizaremos la clase TfidfVectorizer para transformar texto en una representación numérica utilizando el esquema TF-IDF (Term Frequency-Inverse Document Frequency).

Nuestro código es el siguiente:

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:
tfidf = TfidfVectorizer(stop_words='english')
train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

TfidfVectorizer es una clase utilizada para convertir una colección de documentos de texto en una matriz de características.

TF-IDF es una técnica que combina dos estadísticas:

Term Frequency (TF): Frecuencia de un término en un documento, que mide la importancia del término en el documento en cuestión.

Inverse Document Frequency (IDF): Mide la importancia del término en lque a colección de documentos. Penaliza los términos que son muy comunes en todos los documentos.

Llama la atención el argumento “stop_words”. Este arugmento se utilizara para indicar que se deben eliminar las palabras comunes en inglés (como "the", "and", "is", etc.) que no aportan información relevante para el análisis.

Inspeccionemos un poco nuestros objetos antes y después de transformar:

In [13]:
train_x.shape

(6700,)

Nada que nos sorprenda aquí. En train_x tenemos 6,700 reseñas en formato de texto. Ahora, ya tenemos una nueva variable train_x_vector, el cual resultó de ejectutar fit_transform sobre train_x.

In [14]:
train_x_vector.shape

(6700, 44107)

Resulta que ahora tenemos una matriz gigantesca. Esta matriz tiene 6,700 filas y 44,107 columnas. 

Inspeccionemos esto más de cerca:

In [15]:
type(train_x_vector)

scipy.sparse._csr.csr_matrix

El tipo de dato scipy.sparse._csr.csr_matrix es una representación eficiente de una matriz dispersa en formato CSR (Compressed Sparse Row), que se utiliza para almacenar y manipular matrices grandes y dispersas.

Una matriz dispersa es una matriz en la que la mayoría de los elementos son cero. Almacenar todos estos ceros de forma explícita sería ineficiente tanto en términos de espacio como de tiempo. Por eso, en lugar de almacenar la matriz completa, se utiliza una representación que solo guarda los elementos no nulos y su posición, lo que reduce significativamente el uso de memoria y mejora la eficiencia de las operaciones.

Pandas tiene una forma de trabajar con matrices dispersas de tipo csr_matrix:

In [40]:
pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out())

,00,000,007,00am,00s,01,01pm,02,04,05,...,émigré,émigrés,était,étc,être,ísnt,île,önsjön,über,überwoman
6746,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8499,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7869,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3725,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
358,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
761,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]

In [18]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

In [19]:
primera_resenia[primera_resenia != 0]

00           NaN
000          NaN
007          NaN
00am         NaN
00s          NaN
            ... 
ísnt         NaN
île          NaN
önsjön       NaN
über         NaN
überwoman    NaN
Name: 6746, Length: 44107, dtype: Sparse[float64, nan]

In [20]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

## Entranamiento

Usaremos un modelo llamado Máquinas de Soporte Vectorial o Máquinas de Vectores de Soporte.

In [21]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [22]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))

['positive']
['positive']
['negative']


## Evaluación

### Precisión

In [23]:
print(svc.score(test_x_vector, test_y))

0.8706060606060606


### F1 Score

In [24]:
from sklearn.metrics import f1_score

f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

array([0.87400413, 0.86701962])

### Reporte de clasificación

In [25]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                            svc.predict(test_x_vector),
                            labels = ['positive','negative']))

              precision    recall  f1-score   support

    positive       0.87      0.88      0.87      1678
    negative       0.88      0.86      0.87      1622

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300



### Matriz de confusión

In [26]:
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat

array([[1481,  197],
       [ 230, 1392]])

## Modelo: LogisticRegression

In [42]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(train_x_vector, train_y)


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [43]:
print(log_reg.predict(tfidf.transform(['A good movie'])))
print(log_reg.predict(tfidf.transform(['An excellent movie'])))
print(log_reg.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))


['positive']
['positive']
['negative']


In [46]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# Accuracy (score)
print("Accuracy:", log_reg.score(test_x_vector, test_y))

# Precision y F1
y_pred = log_reg.predict(test_x_vector)
print("F1-score:", f1_score(test_y, y_pred, labels = ['positive','negative'],average=None))


# Reporte de clasificación
print("\nClassification Report:\n", classification_report(test_y, y_pred))

# Matriz de confusión
print("\nConfusion Matrix:\n", confusion_matrix(test_y, y_pred))


Accuracy: 0.8718181818181818
F1-score: [0.87488909 0.86859273]

Classification Report:
               precision    recall  f1-score   support

    negative       0.88      0.86      0.87      1622
    positive       0.87      0.88      0.87      1678

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300


Confusion Matrix:
 [[1398  224]
 [ 199 1479]]


El modelo muestra un desempeño sólido:

Accuracy ≈ 87% indica que clasifica correctamente la gran mayoría de reseñas.

F1 ≈ 0.87 en ambas clases refleja equilibrio entre precisión y recall, sin favorecer desproporcionadamente a positivos o negativos.

El reporte de clasificación confirma que tanto la clase negativa como la positiva tienen métricas muy similares, lo que sugiere un modelo balanceado.

La matriz de confusión (1398 aciertos negativos, 1479 aciertos positivos) evidencia que los errores están distribuidos de forma moderada (224 falsos positivos y 199 falsos negativos), sin sesgo fuerte hacia una clase.